# 38 · One Regression, a model per lag, against pandas lags and statsmodels

**Question.** Fieldwork experiment 185 replaced a Relate and a Regression per lag with one of each. Relate brings a
column once per shift; Regression fits the same model once per shifted column, each on its own rows, and an adjustment
can follow the shift: the temperature 2 weeks before is adjusted for the rain 2 weeks before. Does each model give
statsmodels' coefficients and Pearson-scaled standard errors, on the rows that lag alone has?

Check 37's two municipalities of Rio de Janeiro state, 2010 to 2025, the twelve doubled year-edge weeks left out: weekly
cases on the dataset's weekly air temperature at 0, 14 and 28 days before and 14 days after, each adjusted for the
weekly rainfall at the same shift, weeks compared within a municipality, year and month.

**Independent where it matters.** pandas builds each lagged column by merging each week with the same municipality's
week starting that many days later. statsmodels fits each lag on its complete rows with one indicator per stratum.
Neither reads Fieldwork's shifted table.


In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas", "statsmodels"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")

## Parity

Each model's rows and strata, then each term's coefficient and standard error. A model resting on the rows complete at
every lag would use fewer rows than the fixture's.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_38_lag_models import run

report = run(Path.cwd() / "fixtures" / "lag-models-brazil-001.json")
print("agrees:", report["agrees"])
for c in report["comparisons"]:
    print(f"\n{c['column']} with {c['rain']}: rows {c['counts']['rowsUsed']} (complete at every lag: {c['rowsCompleteAtEveryLag']}),"
          f" strata {c['counts']['strataUsed']}, dropped {c['counts']['strataDropped']}")
    for e in c["estimates"]:
        print(f"  {'OK ' if e['agrees'] else 'DIFF'} {e['term']}: coefficient {e['coefficient'][0]:.10g} / {e['coefficient'][1]:.10g}"
              f", se {e['se'][0]:.8g} / {e['se'][1]:.8g}")
print("\ntable keys:", report["table"]["keys"], "· shift named:", report["table"]["shiftNamed"])


## What this cannot establish

**That these lags, or a model per lag, are the right analysis.** The intervals allow nothing for correlation from week
to week, and none is adjusted for the number of lags. **That the dataset's temperature and rainfall measure what a
mosquito meets.**


In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture     :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("statsmodels :", report["environment"]["statsmodels"])
print("runtime     :", report["environment"]["runtime"])
